In [ ]:
# RUN INSTANCES – MOEA/D vs NSGA-II vs PFG-MOEA/D vs PFG-MOEA/D_VER2
# Visualization: Pareto | Hypervolume | C-metric (1 row)

import os
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
from tqdm import tqdm

from algorithm.moead_solver import MOEADSolver
from algorithm.pfg_moead_solver_stop import PFGMOEADSolverStop
from algorithm.pfg_moead_solver_ver2 import PFGMOEADSolverVer2
from algorithm.nsgaii_solver import NSGA2Solver
from algorithm.pfg_moea_solver import PFGMOEASolver

from metric.hv import hypervolume
from metric.cmetric import c_metric
from model.customer import Customer


# CONFIG
POP_SIZE = 100 
NUM_TRUCKS = 12
SEED = 1

DATA_DIR = "400"
RESULT_ROOT = "result_instance_5algo_with_stop_time_56_data400"
os.makedirs(RESULT_ROOT, exist_ok=True)


# LOAD DATASET
def load_customers_from_csv(csv_path):

    df = pd.read_csv(csv_path)
    customers = {}

    depot = df.iloc[0]

    customers[0] = Customer(
        cid=0,
        x=float(depot["x"]),
        y=float(depot["y"]),
        demand=0.0,
        ready_time=float(depot["open"]),
        due_time=float(depot["close"]),
        service_time=float(depot["servicetime"]),
        drone_serve=False,
        time=0.0
    )

    for idx in range(1, len(df)):

        r = df.iloc[idx]

        customers[idx] = Customer(
            cid=idx,
            x=float(r["x"]),
            y=float(r["y"]),
            demand=float(r["demand"]),
            ready_time=float(r["open"]),
            due_time=float(r["close"]),
            service_time=float(r["servicetime"]),
            drone_serve=bool(r["drone_serve"]),
            time=float(r["time"])
        )

    return customers


# UTILS
def pf_to_array(pf):
    return np.array([[s.makespan, s.carbonEmission] for s in pf])


def plot_hv_area(ax, F, ref, color, label):

    F = F[np.argsort(F[:,0])]
    prev_x = ref[0]

    for x,y in F[::-1]:
        ax.fill_betweenx([y,ref[1]],x,prev_x,color=color,alpha=0.18)
        prev_x = x

    ax.plot([],[],color=color,alpha=0.3,label=label)


# PRINT + SAVE PARETO
def print_and_save_pareto(pf, algo_name, result_dir):

    print("\n"+"="*80)
    print(f"{algo_name} Pareto Front")
    print(f"Pareto size = {len(pf)}")
    print("="*80)

    rows = []

    for i,s in enumerate(pf,1):

        print(f"\n--- Solution {i} ---")
        print(f"Makespan = {s.makespan:.4f}")
        print(f"Carbon   = {s.carbonEmission:.4f}")

        for t,route in enumerate(s.truckRoutes):

            drone_list=[]

            if t < len(s.droneCustomers):
                drone_list = sorted(list(s.droneCustomers[t]))

            print(f"Truck {t+1}: {route}")
            print(f"Drone {t+1}: {drone_list}")

            rows.append({
                "algorithm":algo_name,
                "solution_id":i,
                "truck_id":t+1,
                "makespan":s.makespan,
                "carbon":s.carbonEmission,
                "truck_route":" ".join(map(str,route)),
                "drone_customers":" ".join(map(str,drone_list))
            })

    df = pd.DataFrame(rows)

    csv_path = os.path.join(result_dir,f"pareto_{algo_name}.csv")
    df.to_csv(csv_path,index=False)

    print("\nSaved Pareto tours to:",csv_path)


# RUN ONE INSTANCE 
def run_instance(name, customers):

    print(f"\n========== INSTANCE {name} ==========")

    RESULT_DIR = os.path.join(RESULT_ROOT,f"result_{name}")
    os.makedirs(RESULT_DIR,exist_ok=True)

    random.seed(SEED)
    np.random.seed(SEED)


    # ---------- PFG-MOEA/D (STOP) ----------
    print("Running PFG-MOEA/D...")

    pfg = PFGMOEADSolverStop(
        POP_SIZE,
        None,
        NUM_TRUCKS,
        customers,
        SEED
    )

    pf_p = pfg.run()

    MAX_TIME = pfg.run_time

    print("Detected max_time =",MAX_TIME)

    print_and_save_pareto(pf_p,"PFG_MOEA_D",RESULT_DIR)


    # ---------- NSGA-II ----------
    print("Running NSGA-II...")

    nsgaii = NSGA2Solver(POP_SIZE, MAX_TIME, NUM_TRUCKS, customers, SEED)
    pf_n = nsgaii.run()

    print_and_save_pareto(pf_n,"NSGA_II",RESULT_DIR)


    # ---------- MOEA/D ----------
    print("Running MOEA/D...")

    moead = MOEADSolver(POP_SIZE, MAX_TIME, NUM_TRUCKS, customers, SEED)
    pf_m = moead.run()

    print_and_save_pareto(pf_m,"MOEA_D",RESULT_DIR)


    # ---------- PFG-MOEA/D VER2 ----------
    print("Running PFG-MOEA/D_VER2...")

    pfg2 = PFGMOEADSolverVer2(POP_SIZE, MAX_TIME, NUM_TRUCKS, customers, SEED)
    pf_p2 = pfg2.run()

    print_and_save_pareto(pf_p2,"PFG_MOEA_D_VER2",RESULT_DIR)
    
    # ---------- PFG-MOEA ----------
    print("Running PFG-MOEA...")

    pfg_moea = PFGMOEASolver(POP_SIZE, MAX_TIME, NUM_TRUCKS, customers, SEED)
    pf_pm = pfg_moea.run()

    print_and_save_pareto(pf_pm, "PFG_MOEA", RESULT_DIR)
    
    # ---------- SAVE META INFO (max_time + generations) ----------
    df_meta = pd.DataFrame([
        ["PFG_MOEA_D", MAX_TIME, pfg.generations],
        ["NSGA_II", MAX_TIME, nsgaii.generations],
        ["MOEA_D", MAX_TIME, moead.generations],
        ["PFG_MOEA_D_VER2", MAX_TIME, pfg2.generations],
        ["PFG_MOEA", MAX_TIME, pfg_moea.generations],   
    ], columns=["algorithm","max_time","generations"])

    meta_path = os.path.join(RESULT_DIR,"meta_info.csv")
    df_meta.to_csv(meta_path, index=False)

    print("Saved meta info to:", meta_path)

    arr_m = pf_to_array(pf_m)
    arr_p = pf_to_array(pf_p)
    arr_n = pf_to_array(pf_n)
    arr_p2 = pf_to_array(pf_p2)
    arr_pm = pf_to_array(pf_pm)


    ref = np.array([
        max(arr_m[:,0].max(), arr_p[:,0].max(), arr_p2[:,0].max(), arr_n[:,0].max(), arr_pm[:,0].max())*1.1,
        max(arr_m[:,1].max(), arr_p[:,1].max(), arr_p2[:,1].max(), arr_n[:,1].max(), arr_pm[:,1].max())*1.1
    ])


    hv_m = hypervolume(pf_m,ref)
    hv_p = hypervolume(pf_p,ref)
    hv_n = hypervolume(pf_n,ref)
    hv_p2 = hypervolume(pf_p2,ref)
    hv_pm = hypervolume(pf_pm, ref)

    # ---------- C metric ----------
    c_m_p = c_metric(pf_m,pf_p)
    c_p_m = c_metric(pf_p,pf_m)

    c_m_p2 = c_metric(pf_m,pf_p2)
    c_p2_m = c_metric(pf_p2,pf_m)

    c_m_n = c_metric(pf_m,pf_n)
    c_n_m = c_metric(pf_n,pf_m)

    c_p_p2 = c_metric(pf_p,pf_p2)
    c_p2_p = c_metric(pf_p2,pf_p)

    c_p_n = c_metric(pf_p,pf_n)
    c_n_p = c_metric(pf_n,pf_p)

    c_p2_n = c_metric(pf_p2,pf_n)
    c_n_p2 = c_metric(pf_n,pf_p2)

    # ---- PFG-MOEA (NEW) ----
    c_pm_m = c_metric(pf_pm, pf_m)
    c_m_pm = c_metric(pf_m, pf_pm)

    c_pm_p = c_metric(pf_pm, pf_p)
    c_p_pm = c_metric(pf_p, pf_pm)

    c_pm_p2 = c_metric(pf_pm, pf_p2)
    c_p2_pm = c_metric(pf_p2, pf_pm)

    c_pm_n = c_metric(pf_pm, pf_n)
    c_n_pm = c_metric(pf_n, pf_pm)


    print(f"HV(MOEA/D)           = {hv_m:.6f}")
    print(f"HV(PFG-MOEA/D)       = {hv_p:.6f}")
    print(f"HV(PFG-MOEA/D_VER2)  = {hv_p2:.6f}")
    print(f"HV(NSGA-II)          = {hv_n:.6f}")
    print(f"HV(PFG-MOEA)         = {hv_pm:.6f}")

    print(f"C(MOEA/D,PFG-MOEA/D)        = {c_m_p:.6f}")
    print(f"C(PFG-MOEA/D,MOEA/D)        = {c_p_m:.6f}")

    print(f"C(MOEA/D,PFG-MOEA/D_VER2)   = {c_m_p2:.6f}")
    print(f"C(PFG-MOEA/D_VER2,MOEA/D)   = {c_p2_m:.6f}")

    print(f"C(MOEA/D,NSGA-II)           = {c_m_n:.6f}")
    print(f"C(NSGA-II,MOEA/D)           = {c_n_m:.6f}")

    print(f"C(PFG-MOEA/D,PFG-MOEA/D_VER2) = {c_p_p2:.6f}")
    print(f"C(PFG-MOEA/D_VER2,PFG-MOEA/D) = {c_p2_p:.6f}")

    print(f"C(PFG-MOEA/D,NSGA-II)       = {c_p_n:.6f}")
    print(f"C(NSGA-II,PFG-MOEA/D)       = {c_n_p:.6f}")

    print(f"C(PFG-MOEA/D_VER2,NSGA-II)  = {c_p2_n:.6f}")
    print(f"C(NSGA-II,PFG-MOEA/D_VER2)  = {c_n_p2:.6f}")

    print(f"C(PFG-MOEA,MOEA/D)          = {c_pm_m:.6f}")
    print(f"C(MOEA/D,PFG-MOEA)          = {c_m_pm:.6f}")

    print(f"C(PFG-MOEA,PFG-MOEA/D)      = {c_pm_p:.6f}")
    print(f"C(PFG-MOEA/D,PFG-MOEA)      = {c_p_pm:.6f}")

    print(f"C(PFG-MOEA,PFG-MOEA/D_VER2) = {c_pm_p2:.6f}")
    print(f"C(PFG-MOEA/D_VER2,PFG-MOEA) = {c_p2_pm:.6f}")

    print(f"C(PFG-MOEA,NSGA-II)         = {c_pm_n:.6f}")
    print(f"C(NSGA-II,PFG-MOEA)         = {c_n_pm:.6f}")


    fig, axes = plt.subplots(1,3,figsize=(20,5))


    # ---------- Pareto ----------
    axes[0].scatter(arr_m[:,0],arr_m[:,1],s=35,label="MOEA/D")
    axes[0].scatter(arr_p[:,0],arr_p[:,1],s=35,label="PFG-MOEA/D")
    axes[0].scatter(arr_p2[:,0],arr_p2[:,1],s=35,label="PFG-MOEA/D_VER2")
    axes[0].scatter(arr_n[:,0],arr_n[:,1],s=35,label="NSGA-II")
    axes[0].scatter(arr_pm[:,0],arr_pm[:,1],s=35,label="PFG-MOEA")  # NEW

    axes[0].set_title("Pareto Front")
    axes[0].set_xlabel("Makespan")
    axes[0].set_ylabel("Carbon Emission")
    axes[0].grid(True)
    axes[0].legend()


    # ---------- HV ----------
    axes[1].scatter(arr_m[:,0],arr_m[:,1],s=20)
    axes[1].scatter(arr_p[:,0],arr_p[:,1],s=20)
    axes[1].scatter(arr_p2[:,0],arr_p2[:,1],s=20)
    axes[1].scatter(arr_n[:,0],arr_n[:,1],s=20)
    axes[1].scatter(arr_pm[:,0],arr_pm[:,1],s=20)  # NEW

    plot_hv_area(axes[1],arr_m,ref,"blue","MOEA/D HV")
    plot_hv_area(axes[1],arr_p,ref,"red","PFG-MOEA/D HV")
    plot_hv_area(axes[1],arr_p2,ref,"orange","PFG-MOEA/D_VER2 HV")
    plot_hv_area(axes[1],arr_n,ref,"green","NSGA-II HV")
    plot_hv_area(axes[1],arr_pm,ref,"purple","PFG-MOEA HV")  # NEW

    axes[1].scatter(ref[0],ref[1],c="black",marker="x",s=70)

    axes[1].set_title("Hypervolume")
    axes[1].set_xlabel("Makespan")
    axes[1].grid(True)
    axes[1].legend()


    # ---------- C metric ----------
    labels = [
    "C(M,P)","C(P,M)",
    "C(M,P2)","C(P2,M)",
    "C(M,N)","C(N,M)",
    "C(P,P2)","C(P2,P)",
    "C(P,N)","C(N,P)",
    "C(P2,N)","C(N,P2)",
    "C(PM,M)","C(M,PM)",
    "C(PM,P)","C(P,PM)",
    "C(PM,P2)","C(P2,PM)",
    "C(PM,N)","C(N,PM)"
    ]

    values = [
    c_m_p,c_p_m,
    c_m_p2,c_p2_m,
    c_m_n,c_n_m,
    c_p_p2,c_p2_p,
    c_p_n,c_n_p,
    c_p2_n,c_n_p2,
    c_pm_m,c_m_pm,
    c_pm_p,c_p_pm,
    c_pm_p2,c_p2_pm,
    c_pm_n,c_n_pm
    ]


    bars = axes[2].bar(labels,values)

    for bar in bars:
        axes[2].text(
            bar.get_x()+bar.get_width()/2,
            bar.get_height()+0.02,
            f"{bar.get_height():.2f}",
            ha="center",
            rotation=90
        )

    axes[2].set_ylim(0,1)
    axes[2].set_title("C-metric")
    axes[2].grid(axis="y",linestyle="--",alpha=0.6)


    plt.xticks(rotation=45)
    plt.tight_layout()

    plt.savefig(os.path.join(RESULT_DIR,"comparison.png"),dpi=300)

    plt.show()

ALGO_NAMES = [
    "MOEA_D",
    "NSGA_II",
    "PFG_MOEA_D",
    "PFG_MOEA_D_VER2",
    "PFG_MOEA"
]

def is_instance_done(result_dir):
    required_files = ["comparison.png", "meta_info.csv"]

    for algo in ALGO_NAMES:
        required_files.append(f"pareto_{algo}.csv")

    return all(os.path.exists(os.path.join(result_dir, f)) for f in required_files)

# MAIN LOOP 
INSTANCE_FILES = sorted([
    f for f in os.listdir(DATA_DIR)
    if f.endswith(".csv")
])

print("Total instances:", len(INSTANCE_FILES))

for file in tqdm(INSTANCE_FILES):

    path = os.path.join(DATA_DIR, file)
    name = file.replace(".csv", "")

    result_dir = os.path.join(RESULT_ROOT, f"result_{name}")

    # CHECK
    if is_instance_done(result_dir):
        print(f"Skip {name} (already done)")
        continue

    customers = load_customers_from_csv(path)

    run_instance(name, customers)

Total instances: 60


  0%|          | 0/60 [00:00<?, ?it/s]

Skip h400C1_4_1 (already done)
Skip h400C1_4_10 (already done)
Skip h400C1_4_2 (already done)
Skip h400C1_4_3 (already done)

========== INSTANCE h400C1_4_4 ==========
Running PFG-MOEA/D...
